# DeFakeX — Proper Residual Ablation Study

This is a standalone Kaggle notebook. It does **not** load images or frozen experts.

Attach the dataset/file containing:

`heterogeneous_multiview_features.pt`

Then run every cell from top to bottom.

The notebook independently trains and evaluates three fusion models:

- residual scale `0.00`
- residual scale `0.05`
- residual scale `0.25`

All variants use identical cached features, calibration, normalization, initialization, seed, sampler, optimizer, scheduler, early stopping, and validation-only threshold selection.


In [14]:
# CELL 1 — IMPORTS AND CONFIGURATION

import os
import gc
import copy
import json
import random
import shutil
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import (
    Dataset,
    DataLoader,
    WeightedRandomSampler,
)

from sklearn.metrics import (
    accuracy_score,
    recall_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
)

from sklearn.preprocessing import StandardScaler
from IPython.display import display


SEED = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PIN_MEMORY = torch.cuda.is_available()
NUM_WORKERS = 0

RESIDUAL_SCALES = [0.00, 0.05, 0.25]

FUSION_BATCH_SIZE = 384
EVAL_BATCH_SIZE = 1024
FUSION_EPOCHS = 40
FUSION_LR = 3e-4
FUSION_WEIGHT_DECAY = 1e-4
FUSION_PATIENCE = 7
PROJECTION_DIM = 48

ENTROPY_WEIGHT = 0.01
GRAD_CLIP_NORM = 5.0

UNCERTAIN_LOW = 0.30
UNCERTAIN_HIGH = 0.70

OUTPUT_DIR = Path("/kaggle/working/residual_ablation_study")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def seed_everything(seed=SEED):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything()

print("Device:", DEVICE)
print("Residual scales:", RESIDUAL_SCALES)
print("Output directory:", OUTPUT_DIR)


Device: cuda
Residual scales: [0.0, 0.05, 0.25]
Output directory: /kaggle/working/residual_ablation_study


In [15]:
# CELL 2 — FIND AND LOAD THE SAVED FEATURE CACHE

from pathlib import Path
import torch

CACHE_PATH = Path(
    "/kaggle/input/datasets/kashirhanif/frequency-model-checkpoint/heterogeneous_multiview_features.pt"
)

assert CACHE_PATH.exists(), (
    f"heterogeneous_multiview_features.pt was not found at {CACHE_PATH}. "
    "Upload it as a Kaggle dataset and attach that dataset to this notebook."
)

print(f"Selected feature-cache path:\n  {CACHE_PATH}")

cache = torch.load(
    CACHE_PATH,
    map_location="cpu",
    weights_only=False,
)

required_splits = {
    "train",
    "val",
    "mixed_test",
    "ff_test",
    "celeb_test",
    "phone_test",
}

assert isinstance(cache, dict)
assert required_splits.issubset(cache.keys()), (
    f"Missing cache splits: {required_splits - set(cache.keys())}"
)

print("\nLoaded:", CACHE_PATH)
print("Available splits:", list(cache.keys()))

for split in sorted(required_splits):
    print(
        f"{split:<12}",
        f"rows={len(cache[split]['y']):,}",
        f"logits={cache[split]['expert_logits'].shape}",
        f"freq={cache[split]['freq_embedding'].shape}",
        f"spatial={cache[split]['spatial_embedding'].shape}",
        f"metadata={cache[split]['metadata'].shape}",
    )

Selected feature-cache path:
  /kaggle/input/datasets/kashirhanif/frequency-model-checkpoint/heterogeneous_multiview_features.pt

Loaded: /kaggle/input/datasets/kashirhanif/frequency-model-checkpoint/heterogeneous_multiview_features.pt
Available splits: ['train', 'val', 'mixed_test', 'ff_test', 'celeb_test', 'phone_test']
celeb_test   rows=30,000 logits=(30000, 3) freq=(30000, 1536) spatial=(30000, 1536) metadata=(30000, 10)
ff_test      rows=26,276 logits=(26276, 3) freq=(26276, 1536) spatial=(26276, 1536) metadata=(26276, 10)
mixed_test   rows=16,599 logits=(16599, 3) freq=(16599, 1536) spatial=(16599, 1536) metadata=(16599, 10)
phone_test   rows=264 logits=(264, 3) freq=(264, 1536) spatial=(264, 1536) metadata=(264, 10)
train        rows=206,711 logits=(206711, 3) freq=(206711, 1536) spatial=(206711, 1536) metadata=(206711, 10)
val          rows=44,349 logits=(44349, 3) freq=(44349, 1536) spatial=(44349, 1536) metadata=(44349, 10)


In [16]:
# CELL 3 — CACHE INTEGRITY AND LEAKAGE AUDIT

def path_hashes(paths):
    import hashlib

    return {
        hashlib.sha1(
            str(Path(path).resolve()).encode()
        ).hexdigest()
        for path in paths
    }


for split in sorted(required_splits):
    data = cache[split]
    n = len(data["y"])

    required_fields = {
        "expert_logits",
        "freq_embedding",
        "spatial_embedding",
        "metadata",
        "y",
        "scenarios",
    }

    missing = required_fields - set(data.keys())
    assert not missing, f"{split} missing fields: {missing}"

    assert data["expert_logits"].shape == (n, 3)
    assert len(data["freq_embedding"]) == n
    assert len(data["spatial_embedding"]) == n
    assert len(data["metadata"]) == n
    assert len(data["scenarios"]) == n

    for field in (
        "expert_logits",
        "freq_embedding",
        "spatial_embedding",
        "metadata",
        "y",
    ):
        array = np.asarray(data[field])
        invalid = int((~np.isfinite(array)).sum())
        assert invalid == 0, (
            f"{split}/{field} has {invalid} invalid values"
        )

    labels = Counter(np.asarray(data["y"]).astype(int))
    scenarios = Counter(data["scenarios"])

    print(f"\n{split}:")
    print(" labels:", labels)
    print(" scenarios:", scenarios)


# Path-overlap checks are performed only when paths were saved.
if all(
    "paths" in cache[split]
    for split in required_splits
):
    hashes = {
        split: path_hashes(cache[split]["paths"])
        for split in required_splits
    }

    overlap_pairs = [
        ("train", "val"),
        ("train", "mixed_test"),
        ("train", "ff_test"),
        ("train", "celeb_test"),
        ("train", "phone_test"),
        ("val", "phone_test"),
    ]

    print("\nPath overlap audit:")
    for left, right in overlap_pairs:
        overlap = len(hashes[left] & hashes[right])
        print(f"{left:<12} vs {right:<12}: {overlap}")
        assert overlap == 0
else:
    print(
        "\nPath fields are unavailable; numerical integrity "
        "passed, but path-overlap checks were skipped."
    )

print("\nCache audit passed.")



celeb_test:
 labels: Counter({np.int64(1): 20000, np.int64(0): 10000})
 scenarios: Counter({'deepfake_fake': 20000, 'clean_real': 10000})

ff_test:
 labels: Counter({np.int64(1): 22332, np.int64(0): 3944})
 scenarios: Counter({'deepfake_fake': 22332, 'clean_real': 3944})

mixed_test:
 labels: Counter({np.int64(1): 10449, np.int64(0): 6150})
 scenarios: Counter({'ai_fake': 10449, 'clean_real': 6150})

phone_test:
 labels: Counter({np.int64(0): 264})
 scenarios: Counter({'phone_real': 264})

train:
 labels: Counter({np.int64(1): 158832, np.int64(0): 47879})
 scenarios: Counter({'deepfake_fake': 110086, 'ai_fake': 48746, 'clean_real': 47091, 'phone_real': 788})

val:
 labels: Counter({np.int64(1): 33828, np.int64(0): 10521})
 scenarios: Counter({'deepfake_fake': 23384, 'ai_fake': 10444, 'clean_real': 10259, 'phone_real': 262})

Path overlap audit:
train        vs val         : 0
train        vs mixed_test  : 0
train        vs ff_test     : 0
train        vs celeb_test  : 0
train        v

In [17]:
# CELL 4 — EXPERT TEMPERATURE CALIBRATION AND NORMALIZATION

class TemperatureScaler(nn.Module):
    def __init__(self):
        super().__init__()
        self.log_temperature = nn.Parameter(torch.zeros(1))

    def forward(self, logits):
        temperature = (
            self.log_temperature.exp().clamp(0.05, 20.0)
        )
        return logits / temperature


def fit_temperature(logits, labels):
    x = torch.as_tensor(
        logits,
        dtype=torch.float32,
        device=DEVICE,
    )

    y = torch.as_tensor(
        labels,
        dtype=torch.float32,
        device=DEVICE,
    )

    model = TemperatureScaler().to(DEVICE)

    optimizer = torch.optim.LBFGS(
        model.parameters(),
        lr=0.1,
        max_iter=100,
    )

    loss_function = nn.BCEWithLogitsLoss()

    def closure():
        optimizer.zero_grad()
        loss = loss_function(model(x), y)
        loss.backward()
        return loss

    optimizer.step(closure)

    return float(
        model.log_temperature.exp().detach().cpu()
    )


temperatures = [
    fit_temperature(
        cache["val"]["expert_logits"][:, index],
        cache["val"]["y"],
    )
    for index in range(3)
]

print("Expert temperatures:", temperatures)


def calibrated_logits(split):
    logits = (
        cache[split]["expert_logits"]
        .astype(np.float32)
        .copy()
    )

    for index, temperature in enumerate(temperatures):
        logits[:, index] /= temperature

    return logits


frequency_train = cache["train"][
    "freq_embedding"
].astype(np.float32)

spatial_train = cache["train"][
    "spatial_embedding"
].astype(np.float32)

frequency_mean = frequency_train.mean(
    axis=0,
    keepdims=True,
)

frequency_std = frequency_train.std(
    axis=0,
    keepdims=True,
).clip(1e-6)

spatial_mean = spatial_train.mean(
    axis=0,
    keepdims=True,
)

spatial_std = spatial_train.std(
    axis=0,
    keepdims=True,
).clip(1e-6)

metadata_scaler = StandardScaler().fit(
    cache["train"]["metadata"]
)


def prepare_split(split):
    data = cache[split]

    return {
        "logits": calibrated_logits(split),

        "freq": (
            (
                data["freq_embedding"].astype(np.float32)
                - frequency_mean
            )
            / frequency_std
        ).astype(np.float32),

        "spatial": (
            (
                data["spatial_embedding"].astype(np.float32)
                - spatial_mean
            )
            / spatial_std
        ).astype(np.float32),

        "meta": metadata_scaler.transform(
            data["metadata"]
        ).astype(np.float32),

        "raw_meta": data["metadata"].astype(np.float32),

        "y": data["y"].astype(np.float32),

        "scenarios": np.asarray(data["scenarios"]),
    }


prepared = {
    split: prepare_split(split)
    for split in required_splits
}

print("\nPrepared tensors:")
for split in sorted(required_splits):
    data = prepared[split]
    print(
        f"{split:<12}",
        data["logits"].shape,
        data["freq"].shape,
        data["spatial"].shape,
        data["meta"].shape,
    )


Expert temperatures: [5.746281147003174, 5.97663688659668, 3.817824125289917]

Prepared tensors:
celeb_test   (30000, 3) (30000, 1536) (30000, 1536) (30000, 10)
ff_test      (26276, 3) (26276, 1536) (26276, 1536) (26276, 10)
mixed_test   (16599, 3) (16599, 1536) (16599, 1536) (16599, 10)
phone_test   (264, 3) (264, 1536) (264, 1536) (264, 10)
train        (206711, 3) (206711, 1536) (206711, 1536) (206711, 10)
val          (44349, 3) (44349, 1536) (44349, 1536) (44349, 10)


In [18]:
# CELL 5 — DATASET, MODEL, AND IDENTICAL INITIALIZATION

class FusionDataset(Dataset):
    def __init__(self, data):
        self.logits = torch.from_numpy(data["logits"]).float()
        self.freq = torch.from_numpy(data["freq"]).float()
        self.spatial = torch.from_numpy(data["spatial"]).float()
        self.meta = torch.from_numpy(data["meta"]).float()
        self.raw_meta = torch.from_numpy(
            data["raw_meta"]
        ).float()
        self.labels = torch.from_numpy(data["y"]).float()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        return (
            self.logits[index],
            self.freq[index],
            self.spatial[index],
            self.meta[index],
            self.raw_meta[index],
            self.labels[index],
        )


class ResidualGatedFusion(nn.Module):
    def __init__(
        self,
        frequency_dim,
        spatial_dim,
        metadata_dim,
        projection_dim=48,
        residual_scale=0.25,
    ):
        super().__init__()

        self.residual_scale = float(residual_scale)

        self.frequency_projector = nn.Sequential(
            nn.Linear(frequency_dim, 128),
            nn.GELU(),
            nn.Dropout(0.15),
            nn.Linear(128, projection_dim),
            nn.GELU(),
        )

        self.spatial_projector = nn.Sequential(
            nn.Linear(spatial_dim, 128),
            nn.GELU(),
            nn.Dropout(0.15),
            nn.Linear(128, projection_dim),
            nn.GELU(),
        )

        context_dim = (
            3
            + (2 * projection_dim)
            + metadata_dim
        )

        self.gate = nn.Sequential(
            nn.Linear(context_dim, 96),
            nn.GELU(),
            nn.Dropout(0.20),
            nn.Linear(96, 3),
        )

        self.residual = nn.Sequential(
            nn.Linear(context_dim, 64),
            nn.GELU(),
            nn.Dropout(0.15),
            nn.Linear(64, 1),
        )

    def forward(
        self,
        expert_logits,
        frequency_embedding,
        spatial_embedding,
        normalized_metadata,
        raw_metadata,
    ):
        context = torch.cat(
            [
                expert_logits,
                self.frequency_projector(
                    frequency_embedding
                ),
                self.spatial_projector(
                    spatial_embedding
                ),
                normalized_metadata,
            ],
            dim=1,
        )

        weights = torch.softmax(
            self.gate(context),
            dim=1,
        )

        # Column zero is face availability.
        spatial_available = (
            raw_metadata[:, 0:1].clamp(0, 1)
        )

        availability_mask = torch.cat(
            [
                torch.ones_like(spatial_available),
                spatial_available,
                spatial_available,
            ],
            dim=1,
        )

        weights = weights * availability_mask

        weights = weights / weights.sum(
            dim=1,
            keepdim=True,
        ).clamp_min(1e-6)

        weighted_expert_logit = (
            weights * expert_logits
        ).sum(dim=1)

        residual_logit = self.residual(
            context
        ).squeeze(1)

        final_logit = (
            weighted_expert_logit
            + self.residual_scale * residual_logit
        )

        return final_logit, weights


def build_model(residual_scale):
    return ResidualGatedFusion(
        frequency_dim=prepared["train"]["freq"].shape[1],
        spatial_dim=prepared["train"]["spatial"].shape[1],
        metadata_dim=prepared["train"]["meta"].shape[1],
        projection_dim=PROJECTION_DIM,
        residual_scale=residual_scale,
    ).to(DEVICE)


def build_training_loader(seed):
    scenarios = prepared["train"]["scenarios"]
    counts = Counter(scenarios)

    sample_weights = np.asarray(
        [
            1.0 / counts[scenario]
            for scenario in scenarios
        ],
        dtype=np.float64,
    )

    generator = torch.Generator()
    generator.manual_seed(seed)

    sampler = WeightedRandomSampler(
        weights=torch.as_tensor(
            sample_weights,
            dtype=torch.double,
        ),
        num_samples=len(sample_weights),
        replacement=True,
        generator=generator,
    )

    return DataLoader(
        FusionDataset(prepared["train"]),
        batch_size=FUSION_BATCH_SIZE,
        sampler=sampler,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
    )


def build_evaluation_loader(data):
    return DataLoader(
        FusionDataset(data),
        batch_size=EVAL_BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
    )


# Every variant starts from exactly the same parameters.
seed_everything(SEED)

_initial_model = build_model(0.25)
COMMON_INITIAL_STATE = copy.deepcopy(
    _initial_model.state_dict()
)

del _initial_model
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Common initialization saved.")


Common initialization saved.


In [19]:
# CELL 6 — METRICS, THRESHOLD SELECTION, AND INFERENCE

def calculate_metrics(labels, probabilities, threshold):
    predictions = (
        probabilities >= threshold
    ).astype(int)

    return {
        "accuracy": accuracy_score(
            labels,
            predictions,
        ),

        "macro_f1": f1_score(
            labels,
            predictions,
            average="macro",
            zero_division=0,
        ),

        "real_f1": f1_score(
            labels,
            predictions,
            pos_label=0,
            zero_division=0,
        ),

        "fake_f1": f1_score(
            labels,
            predictions,
            pos_label=1,
            zero_division=0,
        ),

        "real_recall": recall_score(
            labels,
            predictions,
            pos_label=0,
            zero_division=0,
        ),

        "fake_recall": recall_score(
            labels,
            predictions,
            pos_label=1,
            zero_division=0,
        ),

        "auc": (
            roc_auc_score(labels, probabilities)
            if len(np.unique(labels)) == 2
            else np.nan
        ),

        "predictions": predictions,
    }


def scenario_recalls(
    labels,
    probabilities,
    threshold,
    scenarios,
):
    predictions = (
        probabilities >= threshold
    ).astype(int)

    scenarios = np.asarray(scenarios)
    output = {}

    for scenario in sorted(set(scenarios)):
        mask = scenarios == scenario
        scenario_label = int(
            round(labels[mask].mean())
        )

        output[scenario] = recall_score(
            labels[mask],
            predictions[mask],
            pos_label=scenario_label,
            zero_division=0,
        )

    return output


def choose_threshold(
    labels,
    probabilities,
    scenarios,
):
    rows = []

    for threshold in np.linspace(
        0.05,
        0.95,
        361,
    ):
        result = calculate_metrics(
            labels,
            probabilities,
            threshold,
        )

        scenario_results = scenario_recalls(
            labels,
            probabilities,
            threshold,
            scenarios,
        )

        worst_scenario = min(
            scenario_results.values()
        )

        recall_gap = abs(
            result["real_recall"]
            - result["fake_recall"]
        )

        score = (
            0.40 * result["macro_f1"]
            + 0.35 * worst_scenario
            + 0.20 * min(
                result["real_recall"],
                result["fake_recall"],
            )
            - 0.15 * recall_gap
        )

        rows.append({
            "threshold": threshold,
            "score": score,
            **{
                key: value
                for key, value in result.items()
                if key != "predictions"
            },
            "worst_scenario": worst_scenario,
            **{
                f"recall_{key}": value
                for key, value
                in scenario_results.items()
            },
        })

    table = pd.DataFrame(rows)
    selected = table.loc[
        table["score"].idxmax()
    ]

    return selected, table


@torch.inference_mode()
def infer_model(model, data):
    model.eval()

    probabilities = []
    gate_weights = []

    for (
        logits,
        frequency,
        spatial,
        metadata,
        raw_metadata,
        _,
    ) in build_evaluation_loader(data):

        final_logits, weights = model(
            logits.to(DEVICE, non_blocking=True),
            frequency.to(DEVICE, non_blocking=True),
            spatial.to(DEVICE, non_blocking=True),
            metadata.to(DEVICE, non_blocking=True),
            raw_metadata.to(DEVICE, non_blocking=True),
        )

        probabilities.append(
            torch.sigmoid(final_logits)
            .cpu()
            .numpy()
        )

        gate_weights.append(
            weights.cpu().numpy()
        )

    return (
        np.concatenate(probabilities),
        np.concatenate(gate_weights),
    )


print("Metric and inference utilities ready.")


Metric and inference utilities ready.


In [20]:
# CELL 7 — TRAIN THE THREE MODELS INDEPENDENTLY

def train_variant(residual_scale):
    # Reset random streams for a fair comparison.
    seed_everything(SEED)

    model = build_model(residual_scale)

    model.load_state_dict(
        COMMON_INITIAL_STATE,
        strict=True,
    )

    training_loader = build_training_loader(SEED)

    loss_function = nn.BCEWithLogitsLoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=FUSION_LR,
        weight_decay=FUSION_WEIGHT_DECAY,
    )

    scheduler = (
        torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="max",
            factor=0.5,
            patience=2,
            min_lr=1e-6,
        )
    )

    best_state = None
    best_score = -np.inf
    best_epoch = 0
    stale_epochs = 0
    history = []

    for epoch in range(1, FUSION_EPOCHS + 1):
        model.train()

        total_loss = 0.0
        total_rows = 0

        for (
            logits,
            frequency,
            spatial,
            metadata,
            raw_metadata,
            labels,
        ) in training_loader:

            logits = logits.to(
                DEVICE,
                non_blocking=True,
            )

            frequency = frequency.to(
                DEVICE,
                non_blocking=True,
            )

            spatial = spatial.to(
                DEVICE,
                non_blocking=True,
            )

            metadata = metadata.to(
                DEVICE,
                non_blocking=True,
            )

            raw_metadata = raw_metadata.to(
                DEVICE,
                non_blocking=True,
            )

            labels = labels.to(
                DEVICE,
                non_blocking=True,
            )

            optimizer.zero_grad(set_to_none=True)

            final_logits, weights = model(
                logits,
                frequency,
                spatial,
                metadata,
                raw_metadata,
            )

            bce_loss = loss_function(
                final_logits,
                labels,
            )

            gate_entropy = -(
                weights.clamp_min(1e-8)
                * weights.clamp_min(1e-8).log()
            ).sum(dim=1).mean()

            loss = (
                bce_loss
                - ENTROPY_WEIGHT * gate_entropy
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                GRAD_CLIP_NORM,
            )

            optimizer.step()

            batch_rows = len(labels)
            total_loss += loss.item() * batch_rows
            total_rows += batch_rows

        validation_probabilities, validation_weights = (
            infer_model(
                model,
                prepared["val"],
            )
        )

        selected_threshold, _ = choose_threshold(
            prepared["val"]["y"],
            validation_probabilities,
            prepared["val"]["scenarios"],
        )

        validation_score = float(
            selected_threshold["score"]
        )

        scheduler.step(validation_score)

        row = {
            "residual_scale": residual_scale,
            "epoch": epoch,
            "loss": total_loss / max(total_rows, 1),
            "learning_rate": optimizer.param_groups[0]["lr"],
            **selected_threshold.to_dict(),
            "gate_frequency": float(
                validation_weights[:, 0].mean()
            ),
            "gate_original_spatial": float(
                validation_weights[:, 1].mean()
            ),
            "gate_auxiliary_spatial": float(
                validation_weights[:, 2].mean()
            ),
        }

        history.append(row)

        print(
            f"scale={residual_scale:.2f} "
            f"epoch={epoch:02d} "
            f"loss={row['loss']:.4f} "
            f"score={validation_score:.4f} "
            f"macro={row['macro_f1']:.4f} "
            f"real={row['real_recall']:.4f} "
            f"fake={row['fake_recall']:.4f} "
            f"worst={row['worst_scenario']:.4f} "
            f"gates={validation_weights.mean(0).round(3)}"
        )

        if validation_score > best_score:
            best_score = validation_score
            best_epoch = epoch
            best_state = copy.deepcopy(
                model.state_dict()
            )
            stale_epochs = 0
        else:
            stale_epochs += 1

            if stale_epochs >= FUSION_PATIENCE:
                print(
                    f"Early stopping scale="
                    f"{residual_scale:.2f}"
                )
                break

    assert best_state is not None

    model.load_state_dict(
        best_state,
        strict=True,
    )

    validation_probabilities, validation_weights = (
        infer_model(
            model,
            prepared["val"],
        )
    )

    selected_threshold, threshold_table = (
        choose_threshold(
            prepared["val"]["y"],
            validation_probabilities,
            prepared["val"]["scenarios"],
        )
    )

    return {
        "model": model,
        "threshold": float(
            selected_threshold["threshold"]
        ),
        "threshold_row": selected_threshold,
        "threshold_table": threshold_table,
        "history": pd.DataFrame(history),
        "best_epoch": best_epoch,
        "best_validation_score": best_score,
        "validation_weights": validation_weights,
    }


trained_variants = {}

for residual_scale in RESIDUAL_SCALES:
    print("\n" + "=" * 88)
    print(
        f"TRAINING RESIDUAL SCALE "
        f"{residual_scale:.2f}"
    )
    print("=" * 88)

    trained_variants[residual_scale] = (
        train_variant(residual_scale)
    )

    result = trained_variants[residual_scale]

    print(
        f"Completed scale={residual_scale:.2f}: "
        f"best_epoch={result['best_epoch']}, "
        f"threshold={result['threshold']:.4f}, "
        f"validation_score="
        f"{result['best_validation_score']:.4f}"
    )

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("\nAll three variants finished.")



TRAINING RESIDUAL SCALE 0.00
scale=0.00 epoch=01 loss=0.3670 score=0.8634 macro=0.9155 real=0.9330 fake=0.9371 worst=0.8891 gates=[0.287 0.447 0.266]
scale=0.00 epoch=02 loss=0.3437 score=0.8391 macro=0.9282 real=0.9114 fake=0.9583 worst=0.8359 gates=[0.25  0.436 0.274]
scale=0.00 epoch=03 loss=0.3292 score=0.8607 macro=0.9168 real=0.9088 fake=0.9473 worst=0.9087 gates=[0.285 0.432 0.27 ]
scale=0.00 epoch=04 loss=0.3263 score=0.8520 macro=0.9109 real=0.9106 fake=0.9404 worst=0.8856 gates=[0.276 0.444 0.258]
scale=0.00 epoch=05 loss=0.3228 score=0.8669 macro=0.9196 real=0.9201 fake=0.9462 worst=0.9113 gates=[0.294 0.432 0.269]
scale=0.00 epoch=06 loss=0.3220 score=0.8674 macro=0.9204 real=0.9195 fake=0.9472 worst=0.9130 gates=[0.29  0.425 0.279]
scale=0.00 epoch=07 loss=0.3224 score=0.8683 macro=0.9210 real=0.9221 fake=0.9469 worst=0.9120 gates=[0.284 0.442 0.269]
scale=0.00 epoch=08 loss=0.3209 score=0.8671 macro=0.9202 real=0.9194 fake=0.9470 worst=0.9122 gates=[0.292 0.436 0.267]
sc

In [21]:
# CELL 8 — FINAL TEST EVALUATION

split_titles = {
    "mixed_test": "Mixed held-out",
    "ff_test": "Pure FF++",
    "celeb_test": "CelebDF zero-shot",
    "phone_test": "Phone held-out",
}

validation_rows = []
test_rows = []
scenario_gate_rows = []
acceptance_rows = []

for residual_scale, trained in trained_variants.items():
    model = trained["model"]
    threshold = trained["threshold"]
    threshold_row = trained["threshold_row"]

    validation_rows.append({
        "residual_scale": residual_scale,
        "best_epoch": trained["best_epoch"],
        "threshold": threshold,
        "validation_score": (
            trained["best_validation_score"]
        ),
        "validation_macro_f1": float(
            threshold_row["macro_f1"]
        ),
        "validation_real_recall": float(
            threshold_row["real_recall"]
        ),
        "validation_fake_recall": float(
            threshold_row["fake_recall"]
        ),
        "validation_auc": float(
            threshold_row["auc"]
        ),
        "validation_worst_scenario": float(
            threshold_row["worst_scenario"]
        ),
    })

    print("\n" + "=" * 88)
    print(
        f"FINAL EVALUATION — RESIDUAL "
        f"{residual_scale:.2f}"
    )
    print("=" * 88)

    scale_results = {}

    for split, title in split_titles.items():
        data = prepared[split]

        probabilities, weights = infer_model(
            model,
            data,
        )

        result = calculate_metrics(
            data["y"],
            probabilities,
            threshold,
        )

        scale_results[split] = result

        test_rows.append({
            "residual_scale": residual_scale,
            "split": split,
            "threshold": threshold,
            "accuracy": result["accuracy"],
            "macro_f1": result["macro_f1"],
            "real_f1": result["real_f1"],
            "fake_f1": result["fake_f1"],
            "real_recall": result["real_recall"],
            "fake_recall": result["fake_recall"],
            "auc": result["auc"],
            "gate_frequency": float(
                weights[:, 0].mean()
            ),
            "gate_original_spatial": float(
                weights[:, 1].mean()
            ),
            "gate_auxiliary_spatial": float(
                weights[:, 2].mean()
            ),
        })

        scenarios = np.asarray(
            data["scenarios"]
        )

        for scenario in sorted(set(scenarios)):
            mask = scenarios == scenario

            scenario_gate_rows.append({
                "residual_scale": residual_scale,
                "split": split,
                "scenario": scenario,
                "n": int(mask.sum()),
                "frequency": float(
                    weights[mask, 0].mean()
                ),
                "original_spatial": float(
                    weights[mask, 1].mean()
                ),
                "auxiliary_spatial": float(
                    weights[mask, 2].mean()
                ),
            })

        print(
            f"{title:<22} "
            f"Macro-F1={result['macro_f1']:.4f}  "
            f"Real={result['real_recall']:.4f}  "
            f"Fake={result['fake_recall']:.4f}  "
            f"AUC={result['auc']:.4f}"
        )

        print(
            "Confusion matrix:",
            confusion_matrix(
                data["y"],
                result["predictions"],
                labels=[0, 1],
            ).tolist(),
        )

    checks = {
        "FF++ macro-F1 >= 0.90":
            scale_results["ff_test"][
                "macro_f1"
            ] >= 0.90,

        "FF++ real recall >= 0.88":
            scale_results["ff_test"][
                "real_recall"
            ] >= 0.88,

        "FF++ fake recall >= 0.92":
            scale_results["ff_test"][
                "fake_recall"
            ] >= 0.92,

        "CelebDF AUC >= 0.76":
            scale_results["celeb_test"][
                "auc"
            ] >= 0.76,

        "CelebDF real recall >= 0.70":
            scale_results["celeb_test"][
                "real_recall"
            ] >= 0.70,

        "CelebDF fake recall >= 0.70":
            scale_results["celeb_test"][
                "fake_recall"
            ] >= 0.70,

        "Phone real recall >= 0.80":
            scale_results["phone_test"][
                "real_recall"
            ] >= 0.80,
    }

    acceptance_rows.append({
        "residual_scale": residual_scale,
        "passed_gates": int(sum(checks.values())),
        "total_gates": len(checks),
        **checks,
    })


validation_comparison_df = pd.DataFrame(
    validation_rows
)

test_results_df = pd.DataFrame(test_rows)

scenario_gate_weights_df = pd.DataFrame(
    scenario_gate_rows
)

acceptance_gates_df = pd.DataFrame(
    acceptance_rows
)

print("\nVALIDATION COMPARISON")
display(
    validation_comparison_df.sort_values(
        "residual_scale"
    )
)

print("\nFINAL TEST COMPARISON")
display(
    test_results_df.sort_values(
        ["split", "residual_scale"]
    )
)

print("\nACCEPTANCE GATES")
display(
    acceptance_gates_df.sort_values(
        ["passed_gates", "residual_scale"],
        ascending=[False, True],
    )
)

print("\nSCENARIO-LEVEL GATE WEIGHTS")
display(
    scenario_gate_weights_df.sort_values(
        ["split", "scenario", "residual_scale"]
    )
)



FINAL EVALUATION — RESIDUAL 0.00
Mixed held-out         Macro-F1=0.9214  Real=0.9553  Fake=0.9074  AUC=0.9782
Confusion matrix: [[5875, 275], [968, 9481]]
Pure FF++              Macro-F1=0.9132  Real=0.8669  Fake=0.9707  AUC=0.9841
Confusion matrix: [[3419, 525], [654, 21678]]
CelebDF zero-shot      Macro-F1=0.6865  Real=0.5883  Fake=0.7862  AUC=0.7488
Confusion matrix: [[5883, 4117], [4276, 15724]]
Phone held-out         Macro-F1=0.4688  Real=0.8826  Fake=0.0000  AUC=nan
Confusion matrix: [[233, 31], [0, 0]]

FINAL EVALUATION — RESIDUAL 0.05
Mixed held-out         Macro-F1=0.9417  Real=0.9712  Fake=0.9290  AUC=0.9901
Confusion matrix: [[5973, 177], [742, 9707]]
Pure FF++              Macro-F1=0.9080  Real=0.8654  Fake=0.9674  AUC=0.9803
Confusion matrix: [[3413, 531], [728, 21604]]
CelebDF zero-shot      Macro-F1=0.6841  Real=0.5565  Fake=0.8066  AUC=0.7473
Confusion matrix: [[5565, 4435], [3867, 16133]]
Phone held-out         Macro-F1=0.4854  Real=0.9432  Fake=0.0000  AUC=nan
Confus

,residual_scale,best_epoch,threshold,validation_score,validation_macro_f1,validation_real_recall,validation_fake_recall,validation_auc,validation_worst_scenario
0,0.00,17,0.5325,0.869754,0.922173,0.924817,0.947174,0.978303,0.912214
1,0.05,8,0.8575,0.883227,0.930927,0.934132,0.953057,0.981268,0.933912
2,0.25,6,0.7975,0.883442,0.930792,0.934892,0.952643,0.981765,0.933742



FINAL TEST COMPARISON


,residual_scale,split,threshold,accuracy,macro_f1,real_f1,fake_f1,real_recall,fake_recall,auc,gate_frequency,gate_original_spatial,gate_auxiliary_spatial
2,0.00,celeb_test,0.5325,0.720233,0.686499,0.583660,0.789338,0.588300,0.786200,0.748787,0.000461,0.506154,0.493386
6,0.05,celeb_test,0.8575,0.723267,0.684061,0.572767,0.795356,0.556500,0.806650,0.747260,0.273003,0.306080,0.420917
10,0.25,celeb_test,0.7975,0.727433,0.687185,0.574978,0.799392,0.553100,0.814600,0.752857,0.284259,0.302351,0.413390
1,0.00,ff_test,0.5325,0.955130,0.913232,0.852938,0.973526,0.866886,0.970715,0.984076,0.000058,0.718366,0.281576
5,0.05,ff_test,0.8575,0.952086,0.907983,0.844280,0.971687,0.865365,0.967401,0.980345,0.316599,0.328808,0.354593
9,0.25,ff_test,0.7975,0.952238,0.908553,0.845348,0.971758,0.869675,0.966819,0.981928,0.325776,0.322842,0.351382
0,0.00,mixed_test,0.5325,0.925116,0.921407,0.904333,0.938481,0.955285,0.907360,0.978218,0.755013,0.026414,0.209841
4,0.05,mixed_test,0.8575,0.944635,0.941684,0.928566,0.954803,0.971220,0.928988,0.990117,0.609531,0.199692,0.190776
8,0.25,mixed_test,0.7975,0.943671,0.940699,0.927424,0.953975,0.971382,0.927361,0.989872,0.563222,0.220122,0.216656
3,0.00,phone_test,0.5325,0.882576,0.468813,0.937626,0.000000,0.882576,0.000000,NaN,0.266140,0.003559,0.691227



ACCEPTANCE GATES


,residual_scale,passed_gates,total_gates,FF++ macro-F1 >= 0.90,FF++ real recall >= 0.88,FF++ fake recall >= 0.92,CelebDF AUC >= 0.76,CelebDF real recall >= 0.70,CelebDF fake recall >= 0.70,Phone real recall >= 0.80
0,0.00,4,7,True,False,True,False,False,True,True
1,0.05,4,7,True,False,True,False,False,True,True
2,0.25,4,7,True,False,True,False,False,True,True



SCENARIO-LEVEL GATE WEIGHTS


,residual_scale,split,scenario,n,frequency,original_spatial,auxiliary_spatial
4,0.00,celeb_test,clean_real,10000,0.000918,0.339703,0.659379
11,0.05,celeb_test,clean_real,10000,0.249103,0.292728,0.458168
18,0.25,celeb_test,clean_real,10000,0.272474,0.295669,0.431857
5,0.00,celeb_test,deepfake_fake,20000,0.000232,0.589379,0.410389
12,0.05,celeb_test,deepfake_fake,20000,0.284952,0.312757,0.402291
19,0.25,celeb_test,deepfake_fake,20000,0.290151,0.305692,0.404157
2,0.00,ff_test,clean_real,3944,0.000235,0.043449,0.956317
9,0.05,ff_test,clean_real,3944,0.271247,0.308511,0.420242
16,0.25,ff_test,clean_real,3944,0.290652,0.314582,0.394766
3,0.00,ff_test,deepfake_fake,22332,0.000027,0.837562,0.162411


In [22]:
# CELL 9 — UNCERTAINTY ANALYSIS

uncertainty_rows = []

for residual_scale, trained in trained_variants.items():
    model = trained["model"]

    for split in (
        "mixed_test",
        "ff_test",
        "celeb_test",
        "phone_test",
    ):
        data = prepared[split]

        probabilities, _ = infer_model(
            model,
            data,
        )

        certain = (
            (probabilities <= UNCERTAIN_LOW)
            | (probabilities >= UNCERTAIN_HIGH)
        )

        certain_accuracy = (
            accuracy_score(
                data["y"][certain],
                (
                    probabilities[certain] >= 0.5
                ).astype(int),
            )
            if certain.any()
            else np.nan
        )

        uncertainty_rows.append({
            "residual_scale": residual_scale,
            "split": split,
            "coverage": float(certain.mean()),
            "certain_accuracy": certain_accuracy,
            "uncertain_rate": float(
                1.0 - certain.mean()
            ),
        })


uncertainty_df = pd.DataFrame(
    uncertainty_rows
)

display(
    uncertainty_df.sort_values(
        ["split", "residual_scale"]
    )
)


,residual_scale,split,coverage,certain_accuracy,uncertain_rate
2,0.00,celeb_test,0.641967,0.777247,0.358033
6,0.05,celeb_test,0.943933,0.749029,0.056067
10,0.25,celeb_test,0.943133,0.751608,0.056867
1,0.00,ff_test,0.934579,0.978418,0.065421
5,0.05,ff_test,0.989877,0.958516,0.010123
9,0.25,ff_test,0.989077,0.958906,0.010923
0,0.00,mixed_test,0.628833,0.981318,0.371167
4,0.05,mixed_test,0.964034,0.971754,0.035966
8,0.25,mixed_test,0.964757,0.970526,0.035243
3,0.00,phone_test,0.643939,0.970588,0.356061


In [23]:
# CELL 10 — CHAMPION RANKING AND SAVE EVERYTHING

for residual_scale, trained in trained_variants.items():
    scale_name = (
        f"{residual_scale:.2f}"
        .replace(".", "p")
    )

    checkpoint = {
        "fusion_state_dict":
            trained["model"].state_dict(),

        "residual_scale":
            residual_scale,

        "threshold":
            trained["threshold"],

        "temperatures":
            temperatures,

        "freq_mean":
            frequency_mean,

        "freq_std":
            frequency_std,

        "spatial_mean":
            spatial_mean,

        "spatial_std":
            spatial_std,

        "metadata_mean":
            metadata_scaler.mean_,

        "metadata_scale":
            metadata_scaler.scale_,

        "projection_dim":
            PROJECTION_DIM,

        "best_epoch":
            trained["best_epoch"],

        "best_validation_score":
            trained["best_validation_score"],

        "uncertain_low":
            UNCERTAIN_LOW,

        "uncertain_high":
            UNCERTAIN_HIGH,

        "seed":
            SEED,

        "training_config": {
            "batch_size": FUSION_BATCH_SIZE,
            "epochs": FUSION_EPOCHS,
            "learning_rate": FUSION_LR,
            "weight_decay": FUSION_WEIGHT_DECAY,
            "patience": FUSION_PATIENCE,
            "entropy_weight": ENTROPY_WEIGHT,
            "gradient_clip_norm": GRAD_CLIP_NORM,
            "sampler":
                "inverse_scenario_frequency",
        },
    }

    torch.save(
        checkpoint,
        OUTPUT_DIR
        / f"defakex_residual_{scale_name}.pth",
    )

    trained["history"].to_csv(
        OUTPUT_DIR
        / (
            f"training_history_"
            f"residual_{scale_name}.csv"
        ),
        index=False,
    )

    trained["threshold_table"].to_csv(
        OUTPUT_DIR
        / (
            f"threshold_sweep_"
            f"residual_{scale_name}.csv"
        ),
        index=False,
    )


validation_comparison_df.to_csv(
    OUTPUT_DIR / "validation_comparison.csv",
    index=False,
)

test_results_df.to_csv(
    OUTPUT_DIR / "test_results.csv",
    index=False,
)

acceptance_gates_df.to_csv(
    OUTPUT_DIR / "acceptance_gates.csv",
    index=False,
)

scenario_gate_weights_df.to_csv(
    OUTPUT_DIR / "scenario_gate_weights.csv",
    index=False,
)

uncertainty_df.to_csv(
    OUTPUT_DIR / "uncertainty_results.csv",
    index=False,
)


# Ranking uses only validation performance and predetermined
# acceptance gates. It never optimizes directly on test metrics.
champion_summary = (
    acceptance_gates_df[
        [
            "residual_scale",
            "passed_gates",
            "total_gates",
        ]
    ]
    .merge(
        validation_comparison_df[
            [
                "residual_scale",
                "validation_score",
                "validation_macro_f1",
                "validation_worst_scenario",
            ]
        ],
        on="residual_scale",
        how="left",
    )
    .sort_values(
        [
            "passed_gates",
            "validation_score",
        ],
        ascending=[False, False],
    )
    .reset_index(drop=True)
)

champion_summary.insert(
    0,
    "rank",
    np.arange(1, len(champion_summary) + 1),
)

champion_summary.to_csv(
    OUTPUT_DIR / "champion_summary.csv",
    index=False,
)

print("Champion ranking:")
display(champion_summary)

print("\nSaved files:")
for path in sorted(OUTPUT_DIR.iterdir()):
    print(" ", path.name)


archive_path = shutil.make_archive(
    "/kaggle/working/defakex_residual_ablation_results",
    "zip",
    OUTPUT_DIR,
)

print("\nResults archive:", archive_path)
print(
    "Download the ZIP and the three .pth checkpoints "
    "before ending the Kaggle session."
)


Champion ranking:


,rank,residual_scale,passed_gates,total_gates,validation_score,validation_macro_f1,validation_worst_scenario
0,1,0.25,4,7,0.883442,0.930792,0.933742
1,2,0.05,4,7,0.883227,0.930927,0.933912
2,3,0.00,4,7,0.869754,0.922173,0.912214



Saved files:
  acceptance_gates.csv
  champion_summary.csv
  defakex_residual_0p00.pth
  defakex_residual_0p05.pth
  defakex_residual_0p25.pth
  scenario_gate_weights.csv
  test_results.csv
  threshold_sweep_residual_0p00.csv
  threshold_sweep_residual_0p05.csv
  threshold_sweep_residual_0p25.csv
  training_history_residual_0p00.csv
  training_history_residual_0p05.csv
  training_history_residual_0p25.csv
  uncertainty_results.csv
  validation_comparison.csv

Results archive: /kaggle/working/defakex_residual_ablation_results.zip
Download the ZIP and the three .pth checkpoints before ending the Kaggle session.


## What to share for review

After the notebook finishes, share either:

- `defakex_residual_ablation_results.zip`, or
- the displayed `validation_comparison`, `test_results`, `acceptance_gates`, and `champion_summary` tables.

Do not replace the existing champion checkpoint until the comparison has been reviewed.
